# Gate 2 · Part 2 — pandas (20 min): a groupby → merge → reshape chain

**The exam-style task.** You're given three tables (described, not shown):

- `inspections`: one row per inspection: `inspection_id`, `camis`, `inspection_date`, `program`, `kind`, `score`
- `violations`: one row per violation cited: `inspection_id`, `violation_code`, `critical_flag` (`"Critical"` / `"Not Critical"` / `"Not Applicable"`)
- `restaurants`: one row per restaurant: `camis`, `boro`, `cuisine`

> **Question:** For each **borough** and **year**, what **percentage of initial cycle inspections cited at least one critical violation**? Return a table with boroughs as rows and years as columns.

Try it on paper first. Then compare with the solution below.

In [1]:
from pathlib import Path
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "P1 - NYC Restaurant Grades").is_dir())
P = REPO / "P1 - NYC Restaurant Grades" / "data" / "processed"
inspections = pd.read_csv(P / "inspections.csv", parse_dates=["inspection_date"])
violations = pd.read_csv(P / "violations.csv")
restaurants = pd.read_csv(P / "restaurants.csv")
print(len(inspections), len(violations), len(restaurants))

135864 294914 25744


## The answer (one chain)

In [2]:
critical_by_inspection = (
    violations
    .assign(is_critical=lambda d: d["critical_flag"].eq("Critical"))
    .groupby("inspection_id", as_index=False)["is_critical"].any()          # 1. GROUPBY: one row per inspection
    .rename(columns={"is_critical": "any_critical"})
)

answer = (
    inspections
    .query("program == 'Cycle Inspection' and kind == 'Initial Inspection'")  # 2. filter to the population
    .merge(critical_by_inspection, on="inspection_id", how="left",
           validate="one_to_one")                                              # 3. MERGE #1 (keep inspections with no violations)
    .assign(any_critical=lambda d: d["any_critical"].eq(True))                  #    no violation rows (NaN) -> no critical violation
    .merge(restaurants[["camis", "boro"]], on="camis", how="left",
           validate="many_to_one")                                             # 4. MERGE #2: add the borough
    .assign(year=lambda d: d["inspection_date"].dt.year)
    .groupby(["boro", "year"])["any_critical"].mean()                          # 5. GROUPBY: share per borough-year
    .mul(100).round(1)
    .unstack("year")                                                           # 6. RESHAPE: years become columns
)
answer

year,2015,2016,2017,2018
boro,,,,
BRONX,84.2,80.8,79.8,81.2
BROOKLYN,84.2,83.7,84.3,83.0
MANHATTAN,83.5,83.1,82.4,82.5
QUEENS,85.0,82.3,82.5,83.1
STATEN ISLAND,85.9,83.7,87.1,83.4


## Why each step is there

| Step | What | Why it matters |
| :-: | :-- | :-- |
| 1 | `groupby(...).any()` on violations | the question is about **inspections**, not violations. Aggregate first, so the merge is one-to-one and nothing gets double-counted |
| 2 | `query(...)` | restrict to the population asked about *before* merging (smaller, faster, clearer) |
| 3 | `merge(how="left", validate="one_to_one")` | **left**: an inspection with no violation rows still counts (as "no critical"). An inner join would silently drop clean inspections and inflate the percentage. `validate` makes pandas raise an error if the keys aren't unique |
| 4 | `merge(validate="many_to_one")` | many inspections per restaurant, one row per restaurant in `restaurants` |
| 5 | `groupby(["boro", "year"]).mean()` | the mean of a True/False column is the share of True |
| 6 | `unstack("year")` | long → wide: one row per borough, one column per year |

## Checks you should always run (merge hygiene from the pandas notebook 04)

In [3]:
pop = inspections.query("program == 'Cycle Inspection' and kind == 'Initial Inspection'")
merged = pop.merge(critical_by_inspection, on="inspection_id", how="left",
                   validate="one_to_one", indicator=True)
print("rows before merge:", len(pop), "| after:", len(merged))
print(merged["_merge"].value_counts())

rows before merge: 70161 | after: 70161
_merge
both          69715
left_only       446
right_only        0
Name: count, dtype: int64


`left_only` rows are inspections with no violation rows in the data. Keeping them is the point of the left join; with an inner join the answer would be wrong. How wrong?

In [4]:
wrong = (pop.merge(critical_by_inspection, on="inspection_id", how="inner")
            .merge(restaurants[["camis", "boro"]], on="camis")
            .assign(year=lambda d: d["inspection_date"].dt.year)
            .groupby(["boro", "year"])["any_critical"].mean().mul(100).round(1).unstack("year"))
(wrong - answer).round(1)   # percentage points the inner join adds

year,2015,2016,2017,2018
boro,,,,
BRONX,0.2,1.3,0.4,0.4
BROOKLYN,0.3,0.5,0.6,0.5
MANHATTAN,0.4,0.3,0.7,0.5
QUEENS,0.7,0.8,0.5,0.4
STATEN ISLAND,0.6,0.4,0.0,0.2


Here the damage is small (0 to 1.3 points) only because just 446 of 70,161 inspections (0.6%) have no violation rows. On data with many clean inspections the same mistake would be large, and nothing would warn you. That's why the join type is a decision, not a default.

## The same answer with `pivot_table` (the other reshape you could use)

In [5]:
alt = (pop.merge(critical_by_inspection, on="inspection_id", how="left", validate="one_to_one")
          .assign(any_critical=lambda d: d["any_critical"].eq(True))
          .merge(restaurants[["camis", "boro"]], on="camis", how="left", validate="many_to_one")
          .assign(year=lambda d: d["inspection_date"].dt.year, any_critical=lambda d: d["any_critical"] * 100.0)
          .pivot_table(index="boro", columns="year", values="any_critical", aggfunc="mean")
          .round(1))
print("identical to the unstack answer:", alt.equals(answer))

identical to the unstack answer: True


## Reading the result (one sentence, as you would in the exam)

About 80-87% of initial inspections cite at least one critical violation, in every borough and every year. Differences between boroughs and years are small (a few points); 2015 has fewer inspections in the data (the 3-year retention window), so compare it with care.

**Caveat specific to this data:** P1's file is a random sample of violation *rows*, so a few critical violations are missing from the sample and these percentages are slightly **too low**. The comparison between boroughs and years is still fair, because the sampling is the same everywhere.